In [2]:
import json
import spacy
from transformers import pipeline

# 1. Load NLP Models
print("Loading NLP models via uv environment...")
nlp = spacy.load("en_core_web_sm")

# HuggingFace Transformers pipelines for zero-shot classification and summarization
classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-3")
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

# 2. Sample Institutional Input Notice
sample_notice = """
CIRCULAR: END-SEMESTER EXAMINATION REGISTRATION 2026
Date: September 20, 2026
To: All Final-Year B.Tech Students

All final-year students are hereby notified that the online registration for the End-Semester Examination 
will open on September 25, 2026. The absolute deadline to submit the registration form and pay the exam fee is 
October 10, 2026. Late submissions will attract a penalty fine. 

Students must upload their fee receipts to the student portal. Course instructors are requested to verify 
the eligibility lists by October 12, 2026. 

For any administrative queries regarding registration, please contact the Academic Controller's office at 
examadmin@university.edu.
"""

# 3. Step A: Classification Engine (Category & Audience)
categories = ["Academic", "Administrative", "Hostel", "Placement", "Exams"]
audiences = ["Students", "Faculty", "Staff"]

cat_res = classifier(sample_notice, candidate_labels=categories)
aud_res = classifier(sample_notice, candidate_labels=audiences)

pred_category = cat_res["labels"][0]
pred_audience = [label for label, score in zip(aud_res["labels"], aud_res["scores"]) if score > 0.35]

# 4. Step B: Information Extraction / NER Engine
doc = nlp(sample_notice)
dates = [ent.text for ent in doc.ents if ent.label_ in ["DATE", "TIME"]]
emails = [token.text for token in doc if token.like_email]

# Rule/keyword-based action item extraction
action_sentences = [
    sent.text.strip() 
    for sent in doc.sents 
    if any(k in sent.text.lower() for k in ["must", "required", "deadline", "submit", "verify"])
]

# 5. Step C: Controlled Persona Summarization
student_summary = summarizer(sample_notice, max_length=45, min_length=15, do_sample=False)[0]["summary_text"]
faculty_summary = summarizer(f"Faculty actionable updates: {sample_notice}", max_length=45, min_length=15, do_sample=False)[0]["summary_text"]

# 6. Step D: Package into Structured JSON Payload
output_payload = {
    "notice_metadata": {
        "category": pred_category,
        "target_audience": pred_audience,
        "urgency": "HIGH" if "deadline" in sample_notice.lower() else "MEDIUM"
    },
    "extracted_entities": {
        "deadlines_and_dates": dates,
        "contact_emails": emails,
        "action_required": action_sentences
    },
    "summaries": {
        "student_view": student_summary,
        "faculty_view": faculty_summary
    }
}

print("\n--- VERSION 1 PIPELINE TEST OUTPUT ---")
print(json.dumps(output_payload, indent=2))

Loading NLP models via uv environment...


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

c:\ProjectFiles\uni-comms-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\a_sri\.cache\huggingface\hub\models--valhalla--distilbart-mnli-12-3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\ProjectFiles\uni-comms-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.p

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.02GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

c:\ProjectFiles\uni-comms-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\a_sri\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"